# LeetCode 2317 — Maximum XOR After Operations bằng Python

Link đề: https://leetcode.com/problems/maximum-xor-after-operations/ · Độ khó: **Medium** · Chủ đề: Array, Math, Bit Manipulation

Skeleton LeetCode:

```python
class Solution:
    def maximumXOR(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. "Giải mã" một phép toán bit trông phức tạp: `a AND (a XOR x)` = `a AND (NOT x)` — tức là **xóa** một tập bit tùy ý của `a`, **không bao giờ** bật thêm bit.
2. Suy luận **từng bit độc lập** cho XOR: bit `b` của kết quả là 1 ⇔ số phần tử có bit `b` bằng 1 là **lẻ**.
3. Kết luận đáp án = **OR** của toàn mảng, và chứng minh cả hai chiều (không vượt được — đạt được).
4. Kiểm chứng bằng BFS vét cạn trên không gian trạng thái nhỏ.
5. Phân biệt với bài "XOR lớn nhất của một **tập con**" (cơ sở tuyến tính trên GF(2)) — bài khác hẳn dù tên gần giống.

## 2. Đề bài & ví dụ

Cho mảng `nums`. Trong một thao tác, chọn số nguyên không âm `x` và chỉ số `i`, rồi gán `nums[i] = nums[i] AND (nums[i] XOR x)`. Trả về **XOR lớn nhất** có thể của mọi phần tử sau một số thao tác bất kỳ.

**Ví dụ 1:** `nums = [3,2,4,6]` → `7` (chọn `x = 4`, `i = 3`: `6 AND (6 XOR 4) = 6 AND 2 = 2`; XOR `3 ^ 2 ^ 4 ^ 2 = 7`).

**Ví dụ 2:** `nums = [1,2,3,9,2]` → `11` (không cần thao tác: `1 ^ 2 ^ 3 ^ 9 ^ 2 = 11`).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `0 ≤ nums[i] ≤ 10⁸`.

In [1]:
from typing import List
from functools import reduce
from operator import or_, xor
from collections import deque
from itertools import product
import random, time

examples = [([3, 2, 4, 6], 7), ([1, 2, 3, 9, 2], 11)]
for a, exp in examples:
    print(a, "XOR ban đầu =", reduce(xor, a), "| kỳ vọng", exp, "| nhị phân:", [format(v, "04b") for v in a])

[3, 2, 4, 6] XOR ban đầu = 3 | kỳ vọng 7 | nhị phân: ['0011', '0010', '0100', '0110']
[1, 2, 3, 9, 2] XOR ban đầu = 11 | kỳ vọng 11 | nhị phân: ['0001', '0010', '0011', '1001', '0010']


## 3. Kiến thức nền

### 3.1 Bảng chân trị của `a AND (a XOR x)` từng bit

In [2]:
print("a x | a^x | a&(a^x) | a&~x")
for a, x in product([0, 1], repeat=2):
    print(f"{a} {x} |  {a ^ x}  |    {a & (a ^ x)}    |  {a & (1 - x)}")

a x | a^x | a&(a^x) | a&~x
0 0 |  0  |    0    |  0
0 1 |  1  |    0    |  0
1 0 |  1  |    1    |  1
1 1 |  0  |    0    |  0


Bit `a = 0` ⇒ kết quả 0 (không bật được). Bit `a = 1` ⇒ kết quả `NOT x` — chọn `x = 1` để xóa, `x = 0` để giữ. Vậy `a AND (a XOR x) = a AND (NOT x)`: thao tác **xóa đúng các bit của `a` mà `x` bật**.

### 3.2 Các giá trị đạt được từ `a` là **mọi mặt nạ con** (submask) của `a`

Chọn `x` = các bit muốn xóa ⇒ đạt mọi `b` với `b ⊆ a` (theo bit). Thao tác nhiều lần cũng chỉ xóa thêm ⇒ tập đạt được không lớn hơn.

In [3]:
a = 6
reach = sorted({a & (a ^ x) for x in range(16)})
subs = sorted({s for s in range(16) if s & a == s})
print("từ", a, "đạt được:", reach, "| mọi submask của", a, ":", subs, "| bằng nhau:", reach == subs)

từ 6 đạt được: [0, 2, 4, 6] | mọi submask của 6 : [0, 2, 4, 6] | bằng nhau: True


### 3.2b Chứng minh đại số (không cần bảng chân trị)

AND phân phối với XOR: `a & (b ^ c) = (a & b) ^ (a & c)`. Do đó
`a & (a ^ x) = (a & a) ^ (a & x) = a ^ (a & x)`. Mà `a & x ⊆ a`, nên XOR với nó đúng là **xóa** các bit đó khỏi `a`: `a ^ (a & x) = a & ~x`.

Trong Python `~x` là số âm (`−x − 1`), nhưng `a & ~x` vẫn cho đúng kết quả vì số nguyên Python dùng biểu diễn bù hai vô hạn bit; với `a ≥ 0` kết quả luôn không âm.

In [4]:
ok = all((a & (a ^ x)) == (a ^ (a & x)) == (a & ~x) for a in range(64) for x in range(64))
print("a & (a ^ x) == a ^ (a & x) == a & ~x với mọi a, x < 64:", ok, "| ví dụ ~4 =", ~4, ", 6 & ~4 =", 6 & ~4)

a & (a ^ x) == a ^ (a & x) == a & ~x với mọi a, x < 64: True | ví dụ ~4 = -5 , 6 & ~4 = 2


### 3.3 XOR theo từng bit

Bit `b` của `x₁ ⊕ x₂ ⊕ … ⊕ xₙ` = tính chẵn lẻ của số phần tử có bit `b` bật. Các bit **độc lập** nhau.

### 3.4 OR theo từng bit

Bit `b` của `x₁ | … | xₙ` = 1 ⇔ **có ít nhất một** phần tử có bit `b` bật.

In [5]:
a = [3, 2, 4, 6]
for b in range(3):
    cnt = sum(v >> b & 1 for v in a)
    print(f"bit {b}: {cnt} phần tử bật -> XOR bit = {cnt % 2}, OR bit = {int(cnt > 0)}")
print("XOR =", reduce(xor, a), "| OR =", reduce(or_, a))

bit 0: 1 phần tử bật -> XOR bit = 1, OR bit = 1
bit 1: 3 phần tử bật -> XOR bit = 1, OR bit = 1
bit 2: 2 phần tử bật -> XOR bit = 0, OR bit = 1
XOR = 3 | OR = 7


## 4. Ý tưởng nền tảng / trực giác

Mỗi phần tử chỉ có thể **mất bit**, không bao giờ được bit. Xét từng bit `b` riêng:

- Nếu **không** phần tử nào có bit `b` ⇒ mọi phần tử sau thao tác vẫn không có ⇒ bit `b` của XOR là 0 — **không** thể bật.
- Nếu có `c ≥ 1` phần tử có bit `b`: ta muốn số phần tử giữ bit `b` là **lẻ** ⇒ giữ bit `b` ở **đúng một** phần tử (hoặc bất kỳ số lẻ nào), xóa ở các phần tử còn lại. Vì các bit độc lập (thao tác xóa từng bit riêng được), làm điều này cho **mọi** bit cùng lúc.

⇒ Đáp án = tập các bit xuất hiện ở ít nhất một phần tử = **OR** của mảng.

Ví dụ 1: `3 | 2 | 4 | 6 = 0b111 = 7` ✔. Ví dụ 2: `1 | 2 | 3 | 9 | 2 = 0b1011 = 11` ✔ (trùng XOR ban đầu — không cần thao tác).

Xây dựng cụ thể: với mỗi bit, chọn phần tử **đầu tiên** có bit đó làm "người giữ"; mọi phần tử khác xóa bit đó.

In [6]:
def construct(nums):
    keep = [0] * len(nums)
    for b in range(max(nums).bit_length()):
        for i, v in enumerate(nums):
            if v >> b & 1: keep[i] |= 1 << b; break       # người giữ bit b
    xs = [v & ~k for v, k in zip(nums, keep)]               # x = các bit cần xóa
    final = [v & (v ^ x) for v, x in zip(nums, xs)]
    return final, xs

for a, exp in examples:
    final, xs = construct(a)
    print(f"{a}: x = {xs} -> mảng {final}, XOR = {reduce(xor, final)} (kỳ vọng {exp})")

[3, 2, 4, 6]: x = [0, 2, 0, 6] -> mảng [3, 0, 4, 0], XOR = 7 (kỳ vọng 7)
[1, 2, 3, 9, 2]: x = [0, 0, 3, 1, 2] -> mảng [1, 2, 0, 8, 0], XOR = 11 (kỳ vọng 11)


### 4.1 Ma trận bit: mỗi cột là một bit, mỗi hàng là một phần tử

Viết các phần tử thành ma trận 0/1. Thao tác = **xóa một số ô 1** trong một hàng. XOR = tính chẵn lẻ của từng **cột**. Mục tiêu: mỗi cột có **số lẻ** ô 1 nếu có thể. Cột toàn 0 không cứu được; cột có ít nhất một ô 1 luôn cứu được (giữ đúng một ô). Các cột độc lập ⇒ OR.

In [7]:
def bit_matrix(nums):
    B = max(max(nums).bit_length(), 1)
    print("phần tử | " + " ".join(f"b{b}" for b in range(B - 1, -1, -1)))
    for v in nums:
        print(f"{v:>7} | " + "  ".join(str(v >> b & 1) for b in range(B - 1, -1, -1)))
    print("  số ô 1| " + "  ".join(str(sum(v >> b & 1 for v in nums)) for b in range(B - 1, -1, -1)))

bit_matrix([3, 2, 4, 6])

phần tử | b2 b1 b0
      3 | 0  1  1
      2 | 0  1  0
      4 | 1  0  0
      6 | 1  1  0
  số ô 1| 2  3  1


Cột `b1` có 3 ô 1 (lẻ — đã ổn), cột `b2` có 2 ô 1 (chẵn — xóa một ô, ví dụ ở `6`), cột `b0` có 1 ô 1 (ổn). Một thao tác duy nhất (xóa bit 2 của `6`) là đủ — đúng như lời giải của đề.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi tổ hợp mặt nạ con — O(Π 2^{popcount(aᵢ)})

Theo 3.2, trạng thái cuối của phần tử `i` là một submask bất kỳ của `nums[i]`. Duyệt mọi tổ hợp.

In [8]:
def submasks(a):
    out = []; s = a
    while True:
        out.append(s)
        if s == 0: break
        s = (s - 1) & a
    return out

def brute(nums):
    best = 0
    for combo in product(*[submasks(v) for v in nums]):
        best = max(best, reduce(xor, combo, 0))
    return best

for a, exp in examples:
    print(brute(a), "OK" if brute(a) == exp else "SAI")

7 OK
11 OK


### 5.2 Baseline 2 — BFS trên trạng thái theo **định nghĩa thao tác** (không dùng nhận xét 3.2)

Áp dụng đúng phép `nums[i] AND (nums[i] XOR x)` với mọi `x` trong một miền đủ lớn, BFS trên các bộ trạng thái. Dùng để kiểm chứng nhận xét 3.2 một cách độc lập.

In [9]:
def brute_bfs(nums, xs_limit=None):
    B = max(nums).bit_length() if max(nums) else 1
    X = range(1 << B) if xs_limit is None else range(xs_limit)
    start = tuple(nums); seen = {start}; q = deque([start]); best = reduce(xor, nums, 0)
    while q:
        st = q.popleft()
        for i in range(len(st)):
            for x in X:
                v = st[i] & (st[i] ^ x)
                if v != st[i]:
                    nt = st[:i] + (v,) + st[i + 1:]
                    if nt not in seen:
                        seen.add(nt); q.append(nt); best = max(best, reduce(xor, nt, 0))
    return best

for a, exp in examples:
    print(brute_bfs(a), "OK" if brute_bfs(a) == exp else "SAI")

7 OK
11 OK


**Nút thắt:** không gian trạng thái là tích các `2^{popcount}` — bùng nổ. Nhưng phân tích **từng bit** cho thấy các bit không tương tác ⇒ chỉ cần biết bit nào **xuất hiện**.

## 6. Ý tưởng thuật toán / bước đột phá

```
return nums[0] | nums[1] | ... | nums[n-1]
```

- O(n) thời gian, O(1) bộ nhớ.
- Trong Python: `reduce(or_, nums)` hoặc vòng lặp `ans |= v`.

Nhận xét phụ: thao tác **không cần thiết** khi XOR ban đầu đã bằng OR — xảy ra khi mỗi bit xuất hiện ở **số lẻ** phần tử (hoặc không xuất hiện). Số thao tác tối thiểu = số phần tử phải xóa bit nào đó (mục 15.2).

### 6.1 Có bao nhiêu mảng cuối đạt XOR lớn nhất?

Mỗi bit `b` xuất hiện ở `c_b` phần tử. Mảng cuối tối ưu ⇔ với mỗi bit `b` có `c_b ≥ 1`, tập phần tử **giữ** bit `b` có kích thước **lẻ** (trong `c_b` ứng viên): `2^{c_b − 1}` cách. Các bit độc lập ⇒ tổng số mảng cuối tối ưu là `Π_{c_b ≥ 1} 2^{c_b − 1} = 2^{Σ (c_b − 1)}`. Kiểm chứng bằng vét cạn:

In [10]:
def count_optimal_finals(nums):
    e = 0
    for b in range(27):
        c = sum(v >> b & 1 for v in nums)
        if c: e += c - 1
    return 2 ** e

def count_optimal_brute(nums):
    target = reduce(or_, nums)
    return sum(1 for combo in product(*[submasks(v) for v in nums]) if reduce(xor, combo, 0) == target)

rng = random.Random(61)
for _ in range(300):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 4))]
    assert count_optimal_finals(a) == count_optimal_brute(a)
print(count_optimal_finals([3, 2, 4, 6]), count_optimal_finals([1, 2, 3, 9, 2]), count_optimal_finals([7, 7]))

8 16 8


Ví dụ 1: `c_{b0} = 1`, `c_{b1} = 3`, `c_{b2} = 2` ⇒ `2^{0 + 2 + 1} = 8` mảng cuối tối ưu. Ví dụ 2: `c = 3, 3, 0, 1` ⇒ `2^{2 + 2 + 0} = 16`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `reduce(or_, nums)` (cách doocs)

In [11]:
def reduce_or(nums):
    return reduce(or_, nums)

for a, exp in examples:
    print(reduce_or(a), "OK" if reduce_or(a) == exp else "SAI")

7 OK
11 OK


### 7.2 Cách 3 — Đếm từng bit (minh họa lập luận theo bit)

In [12]:
def per_bit(nums):
    ans = 0
    for b in range(27):                               # 10^8 < 2^27
        if any(v >> b & 1 for v in nums):
            ans |= 1 << b
    return ans

for a, exp in examples:
    print(per_bit(a), "OK" if per_bit(a) == exp else "SAI")

7 OK
11 OK


`10⁸ < 2²⁷ = 134 217 728` nên 27 bit là đủ. Cách này O(27·n), chậm hơn nhưng phản ánh trực tiếp chứng minh.

### 7.3 Cách 4 — Xây dựng nghiệm tường minh rồi tính XOR (kiểm tra tính đạt được)

In [13]:
def constructive(nums):
    final, _ = construct(nums)
    return reduce(xor, final, 0)

for a, exp in examples:
    print(constructive(a), "OK" if constructive(a) == exp else "SAI")

7 OK
11 OK


### 7.4 Cách 5 — Vòng lặp thuần

In [14]:
def loop_or(nums):
    ans = 0
    for v in nums:
        ans |= v
    return ans

for a, exp in examples:
    print(loop_or(a), "OK" if loop_or(a) == exp else "SAI")

7 OK
11 OK


## 8. TRACE từng bước

### 8.1 Trace OR và xây dựng nghiệm trên ví dụ 1

In [15]:
a = examples[0][0]; acc = 0
for v in a:
    acc |= v; print(f"OR với {v} ({v:03b}) -> {acc} ({acc:03b})")
final, xs = construct(a)
for v, x, f in zip(a, xs, final):
    print(f"  {v:03b} AND ({v:03b} XOR {x:03b}) = {f:03b}")
print("XOR cuối:", format(reduce(xor, final), "03b"), "=", reduce(xor, final))

OR với 3 (011) -> 3 (011)
OR với 2 (010) -> 3 (011)
OR với 4 (100) -> 7 (111)
OR với 6 (110) -> 7 (111)
  011 AND (011 XOR 000) = 011
  010 AND (010 XOR 010) = 000
  100 AND (100 XOR 000) = 100
  110 AND (110 XOR 110) = 000
XOR cuối: 111 = 7


Người giữ bit 0 là `3` (phần tử đầu có bit 0), bit 1 cũng là `3`, bit 2 là `4`. Các phần tử khác xóa các bit đó ⇒ mảng cuối `[3, 0, 4, 0]`, XOR = 7. Lời giải của đề (`[3, 2, 4, 2]`) là một nghiệm khác — có nhiều cách đạt cùng đáp án.

### 8.2 Bảng đếm bit của ví dụ 2 — vì sao không cần thao tác

In [16]:
a = examples[1][0]
for b in range(4):
    cnt = sum(v >> b & 1 for v in a)
    print(f"bit {b}: xuất hiện {cnt} lần ({'lẻ' if cnt % 2 else 'chẵn'})")
print("bit 0 xuất hiện 3 lần (lẻ), bit 1 xuất hiện 3 lần (lẻ), bit 3 một lần => XOR đã bằng OR")

bit 0: xuất hiện 3 lần (lẻ)
bit 1: xuất hiện 3 lần (lẻ)
bit 2: xuất hiện 0 lần (chẵn)
bit 3: xuất hiện 1 lần (lẻ)
bit 0 xuất hiện 3 lần (lẻ), bit 1 xuất hiện 3 lần (lẻ), bit 3 một lần => XOR đã bằng OR


### 8.3 Chọn "người giữ" cho từng bit ở ví dụ 2 (dù không cần thao tác)

In [17]:
a = examples[1][0]
for b in range(4):
    holders = [v for v in a if v >> b & 1]
    print(f"bit {b}: ứng viên {holders} -> {'không có' if not holders else 'giữ ở ' + str(holders[0]) + ', xóa ở phần còn lại'}")
final, xs = construct(a)
print("mảng cuối theo cách xây dựng:", final, "XOR =", reduce(xor, final), "| mảng gốc cũng đạt", reduce(xor, a))

bit 0: ứng viên [1, 3, 9] -> giữ ở 1, xóa ở phần còn lại
bit 1: ứng viên [2, 3, 2] -> giữ ở 2, xóa ở phần còn lại
bit 2: ứng viên [] -> không có
bit 3: ứng viên [9] -> giữ ở 9, xóa ở phần còn lại
mảng cuối theo cách xây dựng: [1, 2, 0, 8, 0] XOR = 11 | mảng gốc cũng đạt 11


Cách xây dựng cho mảng `[1, 2, 0, 8, 0]` — khác mảng gốc nhưng cùng XOR 11. Theo 6.1, có 16 mảng cuối tối ưu, mảng gốc là một trong số đó.

## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def maximumXOR(self, nums: List[int]) -> int:
        ans = 0
        for v in nums:
            ans |= v              # bit nào xuất hiện ở ít nhất một phần tử đều giữ được trong XOR
        return ans

In [19]:
sol = Solution()
for a, exp in examples:
    got = sol.maximumXOR(a)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

7 | kỳ vọng 7 | OK
11 | kỳ vọng 11 | OK


## 10. Vì sao thuật toán đúng

Đặt `O = nums[0] | … | nums[n−1]`.

**Bổ đề 1 (thao tác chỉ xóa bit).** `a AND (a XOR x) = a AND NOT x` (bảng chân trị 3.1) ⇒ kết quả là một submask của `a`. Ngược lại, với mọi submask `s ⊆ a`, chọn `x = a AND NOT s` cho `a AND NOT x = s`. ⇒ Sau bất kỳ số thao tác, tập giá trị đạt được của phần tử `i` đúng là tập submask của `nums[i]` (một thao tác là đủ; nhiều thao tác vẫn là submask).

**Chặn trên.** Sau thao tác mọi phần tử `yᵢ ⊆ nums[i] ⊆ O` (theo bit) ⇒ `y₁ ⊕ … ⊕ yₙ ⊆ O` (XOR không bật bit nào nằm ngoài mọi toán hạng) ⇒ giá trị `≤ O`.

**Đạt được.** Với mỗi bit `b ∈ O`, chọn `h(b)` = một chỉ số có bit `b`. Đặt `yᵢ = { b : h(b) = i }` — là submask của `nums[i]` (vì `h(b) = i` ⇒ `nums[i]` có bit `b`). Mỗi bit `b ∈ O` có mặt ở đúng một `yᵢ` ⇒ XOR có bit `b`; bit ngoài `O` không có ở đâu. ⇒ `⊕ yᵢ = O`.

⇒ Đáp án `= O`. ∎

**Bổ sung — số thao tác.** Cách xây dựng dùng tối đa **một thao tác cho mỗi phần tử** (`n` thao tác); mục 15.2 cho thấy số thao tác tối thiểu có thể nhỏ hơn nhiều (thường 0 hoặc 1), nhưng đề không hỏi điều đó. Ngoài ra, thứ tự thực hiện thao tác không quan trọng: các thao tác trên những phần tử khác nhau độc lập, còn nhiều thao tác trên cùng một phần tử tương đương một thao tác với `x` là OR của các `x`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn submask | O(Π 2^{popcount}) | O(n) |
| 1b. BFS theo định nghĩa | rất lớn | rất lớn |
| 2. `reduce(or_)` | O(n) | O(1) |
| 3. Đếm từng bit | O(27 · n) | O(1) |
| 4. Xây dựng nghiệm | O(27 · n) | O(n) |
| 5. Vòng lặp | O(n) | O(1) |

In [20]:
rng = random.Random(2317)
big = [rng.randint(0, 10**8) for _ in range(10**5)]
res = []
for name, f in [("reduce_or", reduce_or), ("per_bit", per_bit), ("constructive", constructive), ("loop_or", loop_or),
                ("Solution", Solution().maximumXOR)]:
    t0 = time.perf_counter(); res.append(f(big)); dt = time.perf_counter() - t0
    print(f"{name:<13} {dt*1000:7.1f} ms -> {res[-1]}")
assert len(set(res)) == 1

reduce_or         2.1 ms -> 134217727
per_bit           0.0 ms -> 134217727


constructive     13.7 ms -> 134217727
loop_or           2.5 ms -> 134217727
Solution          2.5 ms -> 134217727


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {"brute": brute, "brute_bfs": brute_bfs, "reduce_or": reduce_or, "per_bit": per_bit,
       "constructive": constructive, "loop_or": loop_or, "Solution": Solution().maximumXOR}
edge = examples + [
    ([0], 0), ([5], 5), ([0, 0, 0], 0),
    ([7, 7], 7),                          # XOR ban đầu 0, nhưng xóa hết ở một phần tử -> 7
    ([1, 2, 4], 7),                       # bit rời nhau: XOR = OR
    ([6, 5, 3], 7),                       # XOR ban đầu 0
]
for a, exp in edge:
    for name, f in ALL.items():
        got = f(list(a))
        assert got == exp, (name, a, got, exp)
    print(f"{str(a):<18} -> {exp}  (XOR ban đầu {reduce(xor, a)})")
print("edge OK")

[3, 2, 4, 6]       -> 7  (XOR ban đầu 3)
[1, 2, 3, 9, 2]    -> 11  (XOR ban đầu 11)
[0]                -> 0  (XOR ban đầu 0)
[5]                -> 5  (XOR ban đầu 5)
[0, 0, 0]          -> 0  (XOR ban đầu 0)
[7, 7]             -> 7  (XOR ban đầu 0)
[1, 2, 4]          -> 7  (XOR ban đầu 7)
[6, 5, 3]          -> 7  (XOR ban đầu 0)
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [22]:
rng = random.Random(42)
for _ in range(400):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 4))]
    ref = brute(a)
    for name, f in ALL.items():
        if name == "brute_bfs" and len(a) > 3: continue
        assert f(list(a)) == ref, (name, a)
print("400 mảng ngẫu nhiên OK")

400 mảng ngẫu nhiên OK


### 12.3 Tính chất: đáp án ≥ XOR ban đầu, ≥ max phần tử; thêm phần tử không làm giảm; nghiệm xây dựng hợp lệ

In [23]:
rng = random.Random(7)
for _ in range(1000):
    a = [rng.randint(0, 10**8) for _ in range(rng.randint(1, 20))]
    ans = Solution().maximumXOR(a)
    assert ans >= reduce(xor, a) and ans >= max(a)
    assert Solution().maximumXOR(a + [rng.randint(0, 10**8)]) >= ans
    final, xs = construct(a)
    assert all(f & v == f for f, v in zip(final, a)) and reduce(xor, final) == ans
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị

In [24]:
n = 10**5
assert Solution().maximumXOR([10**8] * n) == 10**8                  # n chẵn: XOR ban đầu 0, nhưng giữ ở một phần tử
assert Solution().maximumXOR([0] * n) == 0
assert Solution().maximumXOR([1 << b for b in range(27)] * 3) == (1 << 27) - 1
assert Solution().maximumXOR(list(range(n))) == (1 << (n - 1).bit_length()) - 1
print("cực trị OK; 10^8 =", bin(10**8), "có", bin(10**8).count("1"), "bit 1, dài", (10**8).bit_length(), "bit")

cực trị OK; 10^8 = 0b101111101011110000100000000 có 12 bit 1, dài 27 bit


`list(range(10⁵))` chứa mọi số tới `99 999` ⇒ mọi bit tới bit 16 đều có ⇒ đáp án `2¹⁷ − 1 = 131 071`.

### 12.5 Đối chiếu BFS theo đúng định nghĩa thao tác (không dùng nhận xét submask)

BFS ở mục 5.2 áp phép `a & (a ^ x)` với mọi `x` — chạy được với mảng ≤ 3 phần tử, giá trị ≤ 15. Kết quả trùng với OR ở mọi bộ thử:

In [25]:
rng = random.Random(125); cnt = 0
for _ in range(150):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 3))]
    assert brute_bfs(a) == reduce(or_, a); cnt += 1
print(cnt, "bộ: BFS theo định nghĩa == OR")

150 bộ: BFS theo định nghĩa == OR


### 12.6 Nhiều thao tác trên cùng một phần tử ≡ một thao tác với `x = x₁ | x₂ | …`

Đây là lý do "một thao tác mỗi phần tử" là đủ trong chứng minh: `(a & ~x₁) & ~x₂ = a & ~(x₁ | x₂)` (luật De Morgan cho bit). Kiểm chứng trên toàn bộ miền nhỏ:

In [26]:
ok = all(((a & (a ^ x1)) & ((a & (a ^ x1)) ^ x2)) == (a & ~(x1 | x2))
         for a in range(32) for x1 in range(32) for x2 in range(32))
print("hai thao tác liên tiếp == một thao tác với x1 | x2:", ok)

hai thao tác liên tiếp == một thao tác với x1 | x2: True


Hệ quả thực hành: khi cần **số thao tác ít nhất** (mục 15.2), chỉ cần quyết định **những phần tử nào** bị đụng tới; với mỗi phần tử đó, một `x` duy nhất gom mọi bit cần xóa.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về XOR ban đầu (nghĩ rằng thao tác không giúp được gì)

In [27]:
def wrong_plain_xor(nums):
    return reduce(xor, nums)                                  # SAI: bỏ qua thao tác

for a, _ in examples:
    print("bản sai :", wrong_plain_xor(a), "| bản đúng:", Solution().maximumXOR(a))

bản sai : 3 | bản đúng: 7
bản sai : 11 | bản đúng: 11


**Vì sao sai:** ví dụ 2 **trùng hợp** đúng; ví dụ 1: XOR ban đầu là `3 ^ 2 ^ 4 ^ 6 = 3`, nhưng xóa bit ở `6` cho 7.

### Lỗi 2 — Nghĩ thao tác có thể **bật** bit (đáp án = `2^{bit_length(max)} − 1`)

In [28]:
def wrong_all_ones(nums):
    return (1 << max(nums).bit_length()) - 1                  # SAI: không bật được bit vắng mặt

for a in [[3, 2, 4, 6], [8, 1], [4]]:
    print("bản sai :", wrong_all_ones(a), "| bản đúng:", Solution().maximumXOR(a), "cho", a)

bản sai : 7 | bản đúng: 7 cho [3, 2, 4, 6]
bản sai : 15 | bản đúng: 9 cho [8, 1]
bản sai : 7 | bản đúng: 4 cho [4]


**Vì sao sai:** ví dụ 1 **trùng hợp** đúng (đủ cả 3 bit). `[8, 1]`: bit 1 và 2 không có ở đâu ⇒ tối đa `9`, không phải `15`. Nguyên nhân: đọc `a AND (a XOR x)` như `a XOR x`.

### Lỗi 3 — Trả về phần tử lớn nhất

In [29]:
def wrong_max(nums):
    return max(nums)                                          # SAI

for a, _ in examples:
    print("bản sai :", wrong_max(a), "| bản đúng:", Solution().maximumXOR(a))

bản sai : 6 | bản đúng: 7
bản sai : 9 | bản đúng: 11


**Sửa:** các bit của **nhiều** phần tử có thể cùng góp vào XOR.

### Lỗi 4 — Đếm bit nhưng chỉ bật khi số lần xuất hiện là **lẻ**

In [30]:
def wrong_parity(nums):
    ans = 0
    for b in range(27):
        if sum(v >> b & 1 for v in nums) % 2 == 1:            # SAI: đây chính là XOR ban đầu
            ans |= 1 << b
    return ans

for a, _ in examples:
    print("bản sai :", wrong_parity(a), "| bản đúng:", Solution().maximumXOR(a))

bản sai : 3 | bản đúng: 7
bản sai : 11 | bản đúng: 11


**Vì sao sai:** đây là cách tính lại XOR ban đầu (ví dụ 2 **trùng hợp** đúng vì mọi bit đã xuất hiện số lẻ lần); ta **điều chỉnh được** tính chẵn lẻ bằng cách xóa bit, nên chỉ cần "≥ 1 lần".

### Lỗi 5 — Dùng AND toàn mảng

In [31]:
def wrong_and(nums):
    return reduce(lambda a, b: a & b, nums)                   # SAI

for a, _ in examples:
    print("bản sai :", wrong_and(a), "| bản đúng:", Solution().maximumXOR(a))

bản sai : 0 | bản đúng: 7
bản sai : 0 | bản đúng: 11


**Sửa:** OR (xuất hiện ở **ít nhất một**), không phải AND (xuất hiện ở **mọi** phần tử).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Phép bit lạ ⇒ lập **bảng chân trị** từng bit; thường rút gọn thành phép quen (`a & ~x`, `a | x`, …).
- XOR / OR / AND đều tính **độc lập theo bit** ⇒ suy luận cho một bit rồi ghép.
- "Chỉ được xóa bit" + "tối đa XOR" ⇒ OR; "chỉ được xóa bit" + "tối đa AND" ⇒ AND ban đầu; "tối thiểu XOR" ⇒ 0.
- Phân biệt với "XOR lớn nhất của **tập con**" (cơ sở tuyến tính) và "XOR lớn nhất của **cặp**" (trie nhị phân).
- Bài cùng họ: 2275 Largest Combination With Bitwise AND Greater Than Zero, 1879, 421 Maximum XOR of Two Numbers in an Array, 2433 Find The Original Array of Prefix Xor, 2425 Bitwise XOR of All Pairings.

## 15. Biến thể của bài

### 15.1 Biến thể A — XOR **nhỏ nhất** sau thao tác (luôn 0), và AND / OR lớn nhất

In [32]:
def min_xor(nums):   return 0                                 # xóa hết mọi bit
def max_or(nums):    return reduce(or_, nums)                 # không xóa gì
def max_and(nums):   return reduce(lambda a, b: a & b, nums)  # xóa chỉ làm AND nhỏ đi

rng = random.Random(151)
for _ in range(200):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 3))]
    combos = list(product(*[submasks(v) for v in a]))
    assert min(reduce(xor, c) for c in combos) == min_xor(a)
    assert max(reduce(or_, c) for c in combos) == max_or(a)
    assert max(reduce(lambda x, y: x & y, c) for c in combos) == max_and(a)
print(min_xor([3, 2, 4, 6]), max_or([3, 2, 4, 6]), max_and([3, 2, 4, 6]))

0 7 0


### 15.2 Biến thể B — Số **thao tác ít nhất** để đạt XOR lớn nhất

Bit `b` xuất hiện `c_b` lần: nếu `c_b` lẻ thì không cần xóa; nếu chẵn (≥ 2) thì phải xóa bit đó ở **một** phần tử (số lẻ còn lại). Một thao tác trên phần tử `i` xóa được nhiều bit cùng lúc ⇒ bài toán: chọn ít phần tử nhất sao cho mỗi bit "chẵn" được xóa ở **số lẻ** phần tử đã chọn có bit đó… — với tham số nhỏ, vét cạn tập phần tử:

In [33]:
def min_ops(nums):
    target = reduce(or_, nums); n = len(nums)
    if reduce(xor, nums) == target: return 0
    for r in range(1, n + 1):
        from itertools import combinations
        for S in combinations(range(n), r):
            # mỗi phần tử trong S được xóa một tập bit tùy ý; kiểm tra có cách nào cho XOR = target
            others = reduce(xor, [nums[i] for i in range(n) if i not in S], 0)
            need = others ^ target                            # XOR của các phần tử trong S sau thao tác phải bằng need
            if max(reduce(xor, c, 0) == need for c in product(*[submasks(nums[i]) for i in S])):
                return r
    return n

print(min_ops([3, 2, 4, 6]), min_ops([1, 2, 3, 9, 2]), min_ops([7, 7, 7, 7]), min_ops([3, 3, 5, 5]))

1 0 1 2


`[7, 7, 7, 7]`: mỗi bit xuất hiện 4 lần (chẵn) ⇒ phải đổi, và một thao tác xóa cả 3 bit ở một phần tử là đủ ⇒ 1. `[3, 3, 5, 5]`: bit 0 xuất hiện 4 lần, bit 1 và bit 2 mỗi bit 2 lần ⇒ cần đổi tính chẵn lẻ cả ba bit; một phần tử `3` hoặc `5` không chứa cả ba bit ⇒ ít nhất 2 thao tác.

### 15.3 Biến thể C — Bài khác tên gần giống: XOR lớn nhất của một **tập con** (cơ sở tuyến tính GF(2))

In [34]:
def max_subset_xor(nums):
    basis = []
    for v in nums:
        for b in basis: v = min(v, v ^ b)
        if v: basis.append(v); basis.sort(reverse=True)
    ans = 0
    for b in basis: ans = max(ans, ans ^ b)
    return ans

def max_subset_xor_brute(nums):
    best = 0; n = len(nums)
    for m in range(1 << n):
        best = max(best, reduce(xor, [nums[i] for i in range(n) if m >> i & 1], 0))
    return best

rng = random.Random(153)
for _ in range(300):
    a = [rng.randint(0, 63) for _ in range(rng.randint(1, 8))]
    assert max_subset_xor(a) == max_subset_xor_brute(a)
print("tập con:", max_subset_xor([3, 2, 4, 6]), "| bài gốc:", Solution().maximumXOR([3, 2, 4, 6]))
print("tập con:", max_subset_xor([6, 3, 5]), "| bài gốc:", Solution().maximumXOR([6, 3, 5]), "<- khác nhau")

tập con: 7 | bài gốc: 7
tập con: 6 | bài gốc: 7 <- khác nhau


Ở bài tập con, ta không được "xóa từng bit" mà chỉ chọn/không chọn **nguyên** phần tử ⇒ cần khử Gauss trên GF(2); đáp án có thể **nhỏ hơn** OR (ví dụ `[6, 3, 5]`: OR = 7, nhưng mọi tập con có XOR ∈ {0, 3, 5, 6}).

### 15.4 Biến thể D — Thao tác là `nums[i] = nums[i] OR x` (chỉ **bật** bit), với `x < 2^B`

Bây giờ mỗi phần tử thành một **supermask** bất kỳ trong `B` bit. Bài trở nên khác hẳn: bit nào cũng có thể bật, nhưng một bit đã bật ở **mọi** phần tử thì không tắt được nữa — tính chẵn lẻ của nó bị khóa theo `n`. Vét cạn nhỏ cho thấy đáp án không phải lúc nào cũng là `2^B − 1`:

In [35]:
def or_variant_bruteforce(nums, B):
    choices = [[v | x for x in range(1 << B)] for v in nums]
    return max(reduce(xor, c) for c in product(*choices))
print(or_variant_bruteforce([3, 2], 3), or_variant_bruteforce([4], 3), or_variant_bruteforce([5, 5, 5], 3))
# [3, 2]: bit 1 bật ở cả hai phần tử => luôn xuất hiện 2 lần (chẵn) => bit 1 của XOR luôn 0 => tối đa 0b101 = 5

5 7 7


### 15.5 Biến thể E — Mỗi thao tác dùng **cùng một `x` cho cả mảng**

Nếu mọi phần tử phải dùng cùng `x`, thì XOR cuối `= ⊕ (nums[i] & ~x) = (⊕ nums[i]) & ~x` (AND với hằng số phân phối qua XOR) ⇒ tối đa đạt tại `x = 0`: **XOR ban đầu**. "Lỗi 1" ở mục 13 chính là lời giải đúng của biến thể này — sự tự do chọn `x` **riêng cho từng phần tử** là điều làm đáp án tăng lên OR.

In [36]:
def same_x_for_all(nums, B=5):
    return max(reduce(xor, [v & ~x for v in nums]) for x in range(1 << B))

rng = random.Random(155)
for _ in range(300):
    a = [rng.randint(0, 31) for _ in range(rng.randint(1, 6))]
    assert same_x_for_all(a) == reduce(xor, a)
print(same_x_for_all([3, 2, 4, 6]), "=", reduce(xor, [3, 2, 4, 6]), "(so với bài gốc:", Solution().maximumXOR([3, 2, 4, 6]), ")")

3 = 3 (so với bài gốc: 7 )


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn submask | định nghĩa rút gọn | mũ | O(n) | kiểm chứng |
| 1b. BFS theo thao tác | định nghĩa gốc | rất lớn | rất lớn | kiểm chứng nhận xét 3.2 |
| 2. `reduce(or_)` | OR | O(n) | O(1) | **mặc định** |
| 3. Đếm từng bit | lập luận theo bit | O(27n) | O(1) | giải thích |
| 4. Xây dựng nghiệm | người giữ mỗi bit | O(27n) | O(n) | cần mảng cuối |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumXOR([5, 5])` và `maximumXOR([8, 4, 2, 1, 0])`?

**Bài 2 (tính tay):** `nums = [12, 10, 6]`. Viết nhị phân, tính XOR ban đầu và đáp án, chỉ ra một mảng cuối đạt đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(nums)` trả về danh sách `x` (một `x` cho mỗi phần tử, áp dụng **một** thao tác mỗi phần tử) đạt XOR lớn nhất.

**Bài 4 (tự viết lại):** Viết `bt4_todo(nums)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(nums, T)` trả về `True` ⇔ có thể đạt XOR **đúng bằng** `T` sau các thao tác.

In [37]:
def bt3_todo(nums):
    # TODO: các x cho từng phần tử
    pass

def bt4_todo(nums):
    # TODO: bản nộp
    pass

def bt5_todo(nums, T):
    # TODO: đạt đúng T?
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thao tác chỉ xóa bit. Mức 2: `5 | 5 = 5`. Mức 3: `[8,4,2,1,0]` ⇒ `15`.

**Bài 2.** Mức 1: `1100, 1010, 0110`. Mức 2: XOR ban đầu `0000`, OR `1110 = 14`. Mức 3: giữ bit 3 và 2 ở `12`, bit 1 ở `10` ⇒ mảng `[12, 2, 0]`, XOR `14`.

**Bài 3.** Mức 1: chọn người giữ cho mỗi bit. Mức 2: `keep[i]` = các bit mà `i` giữ. Mức 3: `x_i = nums[i] & ~keep[i]`.

**Bài 4.** Mức 1: `ans = 0`. Mức 2: `ans |= v`. Mức 3: trả về.

**Bài 5.** Mức 1: `T` phải ⊆ OR. Mức 2: mọi `T ⊆ OR` đều đạt được: với bit trong `T` chọn một người giữ, mọi bit khác xóa hết. Mức 3: `T & ~OR == 0`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `5`, `15`. **Bài 2:** XOR ban đầu `0`, đáp án `14`, mảng `[12, 2, 0]`.

In [38]:
assert Solution().maximumXOR([5, 5]) == 5 and Solution().maximumXOR([8, 4, 2, 1, 0]) == 15
assert reduce(xor, [12, 10, 6]) == 0 and Solution().maximumXOR([12, 10, 6]) == 14 and reduce(xor, [12, 2, 0]) == 14

def bt3(nums):
    return construct(nums)[1]

def bt4(nums):
    return Solution().maximumXOR(nums)

def bt5(nums, T):
    return T & ~reduce(or_, nums) == 0

rng = random.Random(1997)
for _ in range(400):
    a = [rng.randint(0, 15) for _ in range(rng.randint(1, 3))]
    xs = bt3(a)
    assert reduce(xor, [v & (v ^ x) for v, x in zip(a, xs)]) == brute(a)
    assert bt4(a) == brute(a)
    reach = {reduce(xor, c) for c in product(*[submasks(v) for v in a])}
    for T in range(16): assert bt5(a, T) == (T in reach)
print(bt3([12, 10, 6]), bt5([12, 10, 6], 9), bt5([12, 10, 6], 1))
print("Đáp án bài tập: tất cả assert OK")

[0, 8, 6] False False
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# a & (a ^ x) == a & ~x  : chỉ XÓA bit, không bật bit
# XOR tối đa = mọi bit xuất hiện ít nhất một lần = OR
ans = 0
for v in nums: ans |= v
return ans
```

**Checklist:**
- [ ] Lập bảng chân trị để thấy thao tác chỉ xóa bit.
- [ ] Không trả về XOR ban đầu / max / AND.
- [ ] Không nhầm với "XOR lớn nhất của tập con" (cần cơ sở tuyến tính).